In [1]:
import torch
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("Compute capability:", torch.cuda.get_device_capability(0))
    print("Memory:", torch.cuda.get_device_properties(0).total_memory / 1024**3, "GB")

PyTorch: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4
Compute capability: (7, 5)
Memory: 14.56317138671875 GB


In [2]:
import triton
import triton.language as tl
print("Triton:", triton.__version__)

Triton: 3.6.0


In [3]:
!pip install -q triton

In [4]:
import torch
import triton
import triton.language as tl

@triton.jit
def add_kernel(x_ptr, y_ptr, out_ptr, n, BLOCK: tl.constexpr):
    pid = tl.program_id(0)
    offs = pid * BLOCK + tl.arange(0, BLOCK)
    mask = offs < n
    x = tl.load(x_ptr + offs, mask=mask)
    y = tl.load(y_ptr + offs, mask=mask)
    tl.store(out_ptr + offs, x + y, mask=mask)

n = 1024
x = torch.randn(n, device='cuda')
y = torch.randn(n, device='cuda')
out = torch.empty_like(x)
add_kernel[(triton.cdiv(n, 256),)](x, y, out, n, BLOCK=256)
torch.testing.assert_close(out, x + y)
print("Triton работает корректно.")

Triton работает корректно.


In [5]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [6]:
%cd /content
import os
if not os.path.exists('/content/MNNA-2026'):
    !git clone https://github.com/ksenkap/MNNA-2026.git
else:
    print("Репозиторий уже склонирован.")
%cd /content/MNNA-2026
!git fetch --all

/content
Репозиторий уже склонирован.
/content/MNNA-2026


In [18]:
!grep -q "^triton" requirements.txt && echo "triton уже в requirements" || (echo "triton>=3.0.0" >> requirements.txt && echo "triton добавлен")
!tail -5 requirements.txt

triton добавлен
tokenizers>=0.13.0
nltk>=3.8.0
beautifulsoup4>=4.12.0
langdetect>=1.0.9
triton>=3.0.0


In [1]:
%%writefile /content/MNNA-2026/src/backend/flash_attention_forward.py
import torch
import triton
import triton.language as tl
import math


@triton.jit
def _flash_attn_fwd_kernel(
    Q, K, V, Out, L,
    stride_qz, stride_qh, stride_qm, stride_qd,
    stride_kz, stride_kh, stride_kn, stride_kd,
    stride_vz, stride_vh, stride_vn, stride_vd,
    stride_oz, stride_oh, stride_om, stride_od,
    Z, H, N_CTX,
    scale,
    BLOCK_M: tl.constexpr, BLOCK_N: tl.constexpr, BLOCK_DMODEL: tl.constexpr,
    IS_CAUSAL: tl.constexpr,
):
    start_m = tl.program_id(0)
    off_hz = tl.program_id(1)
    off_z = off_hz // H
    off_h = off_hz % H

    offs_m = start_m * BLOCK_M + tl.arange(0, BLOCK_M)
    offs_d = tl.arange(0, BLOCK_DMODEL)

    q_offset = off_z * stride_qz + off_h * stride_qh
    q_ptrs = Q + q_offset + offs_m[:, None] * stride_qm + offs_d[None, :] * stride_qd
    q_mask = offs_m[:, None] < N_CTX
    q = tl.load(q_ptrs, mask=q_mask, other=0.0)

    m_i = tl.full([BLOCK_M], float('-inf'), dtype=tl.float32)
    l_i = tl.zeros([BLOCK_M], dtype=tl.float32)
    acc = tl.zeros([BLOCK_M, BLOCK_DMODEL], dtype=tl.float32)

    if IS_CAUSAL:
        hi = tl.cdiv((start_m + 1) * BLOCK_M, BLOCK_N)
    else:
        hi = tl.cdiv(N_CTX, BLOCK_N)

    k_offset = off_z * stride_kz + off_h * stride_kh
    v_offset = off_z * stride_vz + off_h * stride_vh

    for start_n in range(0, hi * BLOCK_N, BLOCK_N):
        start_n = tl.multiple_of(start_n, BLOCK_N)
        offs_n = start_n + tl.arange(0, BLOCK_N)

        k_ptrs = K + k_offset + offs_n[None, :] * stride_kn + offs_d[:, None] * stride_kd
        k_mask = offs_n[None, :] < N_CTX
        k = tl.load(k_ptrs, mask=k_mask, other=0.0)

        s = tl.dot(q, k) * scale

        if IS_CAUSAL:
            causal_mask = offs_m[:, None] >= offs_n[None, :]
            s = tl.where(causal_mask, s, float('-inf'))

        n_mask = offs_n[None, :] < N_CTX
        s = tl.where(n_mask, s, float('-inf'))

        m_ij = tl.max(s, axis=1)
        m_new = tl.maximum(m_i, m_ij)
        alpha = tl.exp(m_i - m_new)
        p = tl.exp(s - m_new[:, None])
        l_i = l_i * alpha + tl.sum(p, axis=1)
        acc = acc * alpha[:, None]

        v_ptrs = V + v_offset + offs_n[:, None] * stride_vn + offs_d[None, :] * stride_vd
        v_mask = offs_n[:, None] < N_CTX
        v = tl.load(v_ptrs, mask=v_mask, other=0.0)
        acc = tl.dot(p.to(v.dtype), v, acc)

        m_i = m_new

    acc = acc / l_i[:, None]
    L_i = m_i + tl.log(l_i)

    o_offset = off_z * stride_oz + off_h * stride_oh
    o_ptrs = Out + o_offset + offs_m[:, None] * stride_om + offs_d[None, :] * stride_od
    tl.store(o_ptrs, acc.to(Out.dtype.element_ty), mask=q_mask)

    l_ptrs = L + off_hz * N_CTX + offs_m
    tl.store(l_ptrs, L_i, mask=offs_m < N_CTX)


def flash_attn_forward(q, k, v, causal=True):
    """
    q, k, v: (B, H, N, D), dtype fp16
    Возвращает: out (B, H, N, D), L (B, H, N) — logsumexp
    """
    B, H, N, D = q.shape
    assert D in {16, 32, 64, 128, 256}, f"D={D} not supported"
    q, k, v = q.contiguous(), k.contiguous(), v.contiguous()
    out = torch.empty_like(q)
    L = torch.empty((B, H, N), device=q.device, dtype=torch.float32)
    scale = 1.0 / math.sqrt(D)

    # Подбор блоков под SMEM T4 (64 KB)
    if D <= 64:
        BLOCK_M, BLOCK_N = 128, 64
    elif D <= 128:
        BLOCK_M, BLOCK_N = 64, 64
    else:
        BLOCK_M, BLOCK_N = 64, 32

    grid = (triton.cdiv(N, BLOCK_M), B * H)

    _flash_attn_fwd_kernel[grid](
        q, k, v, out, L,
        q.stride(0), q.stride(1), q.stride(2), q.stride(3),
        k.stride(0), k.stride(1), k.stride(2), k.stride(3),
        v.stride(0), v.stride(1), v.stride(2), v.stride(3),
        out.stride(0), out.stride(1), out.stride(2), out.stride(3),
        B, H, N, scale,
        BLOCK_M=BLOCK_M, BLOCK_N=BLOCK_N, BLOCK_DMODEL=D,
        IS_CAUSAL=causal,
    )
    return out, L

Overwriting /content/MNNA-2026/src/backend/flash_attention_forward.py


In [33]:
import sys
sys.path.insert(0, '/content/MNNA-2026')
import importlib
import src.backend.flash_attention_forward as faf
importlib.reload(faf)
from src.backend.flash_attention_forward import flash_attn_forward

import torch
import math


def reference_attention(q, k, v, causal=True):
    """Наивная реализация для сверки. Вход: (B, H, N, D) fp16."""
    B, H, N, D = q.shape
    scale = 1.0 / math.sqrt(D)
    s = (q.float() @ k.float().transpose(-2, -1)) * scale
    if causal:
        mask = torch.tril(torch.ones(N, N, device=q.device, dtype=torch.bool))
        s = s.masked_fill(~mask, float('-inf'))
    p = torch.softmax(s, dim=-1)
    return (p @ v.float()).to(q.dtype)


def test_forward(N, D, causal, B=2, H=4, atol=1e-2, rtol=1e-2):
    torch.manual_seed(0)
    q = torch.randn(B, H, N, D, device='cuda', dtype=torch.float16)
    k = torch.randn(B, H, N, D, device='cuda', dtype=torch.float16)
    v = torch.randn(B, H, N, D, device='cuda', dtype=torch.float16)

    out, L = flash_attn_forward(q, k, v, causal=causal)
    ref = reference_attention(q, k, v, causal=causal)

    torch.testing.assert_close(out, ref, atol=atol, rtol=rtol)
    print(f"  OK: N={N}, D={D}, causal={causal}")


print("=== Forward тесты ===")
# 1. Без causal, один блок
test_forward(N=128, D=64, causal=False)
# 2. Без causal, несколько блоков
test_forward(N=512, D=64, causal=False)
# 3. Causal, один блок
test_forward(N=128, D=64, causal=True)
# 4. Causal, несколько блоков
test_forward(N=512, D=64, causal=True)
# 5. N не кратен BLOCK
test_forward(N=333, D=64, causal=False)
test_forward(N=333, D=64, causal=True)
# 6. D=128
test_forward(N=512, D=128, causal=True)
print("Все тесты forward пройдены.")

=== Forward тесты ===
  OK: N=128, D=64, causal=False
  OK: N=512, D=64, causal=False
  OK: N=128, D=64, causal=True
  OK: N=512, D=64, causal=True
  OK: N=333, D=64, causal=False
  OK: N=333, D=64, causal=True
  OK: N=512, D=128, causal=True
Все тесты forward пройдены.


In [5]:
%%writefile /content/MNNA-2026/src/backend/flash_attention_backward.py
import torch
import triton
import triton.language as tl
import math


# ---------------------------------------------------------------------------
# Кернел 1: dK, dV. Внешний цикл по блокам Q (i), накопление dK, dV.
# ---------------------------------------------------------------------------
@triton.jit
def _flash_attn_bwd_dkdv_kernel(
    Q, K, V, dO, dK, dV, L, D,
    stride_qz, stride_qh, stride_qm, stride_qd,
    stride_kz, stride_kh, stride_kn, stride_kd,
    stride_vz, stride_vh, stride_vn, stride_vd,
    stride_oz, stride_oh, stride_om, stride_od,
    stride_dkz, stride_dkh, stride_dkn, stride_dkd,
    stride_dvz, stride_dvh, stride_dvn, stride_dvd,
    Z, H, N_CTX,
    scale,
    BLOCK_M: tl.constexpr, BLOCK_N: tl.constexpr, BLOCK_DMODEL: tl.constexpr,
    IS_CAUSAL: tl.constexpr,
):
    start_n = tl.program_id(0)
    off_hz = tl.program_id(1)
    off_z = off_hz // H
    off_h = off_hz % H

    offs_n = start_n * BLOCK_N + tl.arange(0, BLOCK_N)
    offs_d = tl.arange(0, BLOCK_DMODEL)

    k_offset = off_z * stride_kz + off_h * stride_kh
    v_offset = off_z * stride_vz + off_h * stride_vh

    k_ptrs = K + k_offset + offs_n[:, None] * stride_kn + offs_d[None, :] * stride_kd
    k_mask = offs_n[:, None] < N_CTX
    k_block = tl.load(k_ptrs, mask=k_mask, other=0.0)

    v_ptrs = V + v_offset + offs_n[:, None] * stride_vn + offs_d[None, :] * stride_vd
    v_mask = offs_n[:, None] < N_CTX
    v_block = tl.load(v_ptrs, mask=v_mask, other=0.0)

    dk = tl.zeros([BLOCK_N, BLOCK_DMODEL], dtype=tl.float32)
    dv = tl.zeros([BLOCK_N, BLOCK_DMODEL], dtype=tl.float32)

    if IS_CAUSAL:
        lo = (start_n * BLOCK_N) // BLOCK_M
    else:
        lo = 0
    hi = tl.cdiv(N_CTX, BLOCK_M)

    q_offset = off_z * stride_qz + off_h * stride_qh
    o_offset = off_z * stride_oz + off_h * stride_oh

    for start_m in range(lo * BLOCK_M, hi * BLOCK_M, BLOCK_M):
        start_m = tl.multiple_of(start_m, BLOCK_M)
        offs_m = start_m + tl.arange(0, BLOCK_M)

        q_ptrs = Q + q_offset + offs_m[:, None] * stride_qm + offs_d[None, :] * stride_qd
        q_mask = offs_m[:, None] < N_CTX
        q = tl.load(q_ptrs, mask=q_mask, other=0.0)

        do_ptrs = dO + o_offset + offs_m[:, None] * stride_om + offs_d[None, :] * stride_od
        do_mask = offs_m[:, None] < N_CTX
        do = tl.load(do_ptrs, mask=do_mask, other=0.0)

        l_ptrs = L + off_hz * N_CTX + offs_m
        l_mask = offs_m < N_CTX
        L_i = tl.load(l_ptrs, mask=l_mask, other=0.0)

        d_ptrs = D + off_hz * N_CTX + offs_m
        delta_i = tl.load(d_ptrs, mask=l_mask, other=0.0)

        s = tl.dot(q, tl.trans(k_block)) * scale

        if IS_CAUSAL:
            causal = offs_m[:, None] >= offs_n[None, :]
            s = tl.where(causal, s, float('-inf'))
        n_mask = offs_n[None, :] < N_CTX
        s = tl.where(n_mask, s, float('-inf'))
        m_mask = offs_m[:, None] < N_CTX
        s = tl.where(m_mask, s, float('-inf'))

        p = tl.exp(s - L_i[:, None])

        dv += tl.dot(tl.trans(p).to(do.dtype), do)

        dp = tl.dot(do, tl.trans(v_block))
        ds = p * (dp - delta_i[:, None])

        dk += tl.dot(tl.trans(ds).to(q.dtype), q) * scale

    dk_offset = off_z * stride_dkz + off_h * stride_dkh
    dk_ptrs = dK + dk_offset + offs_n[:, None] * stride_dkn + offs_d[None, :] * stride_dkd
    tl.store(dk_ptrs, dk.to(dK.dtype.element_ty), mask=k_mask)

    dv_offset = off_z * stride_dvz + off_h * stride_dvh
    dv_ptrs = dV + dv_offset + offs_n[:, None] * stride_dvn + offs_d[None, :] * stride_dvd
    tl.store(dv_ptrs, dv.to(dV.dtype.element_ty), mask=v_mask)


# ---------------------------------------------------------------------------
# Кернел 2: dQ. Внешний цикл по блокам K/V (j), накопление dQ.
# ---------------------------------------------------------------------------
@triton.jit
def _flash_attn_bwd_dq_kernel(
    Q, K, V, dO, dQ, L, D,
    stride_qz, stride_qh, stride_qm, stride_qd,
    stride_kz, stride_kh, stride_kn, stride_kd,
    stride_vz, stride_vh, stride_vn, stride_vd,
    stride_oz, stride_oh, stride_om, stride_od,
    stride_dqz, stride_dqh, stride_dqm, stride_dqd,
    Z, H, N_CTX,
    scale,
    BLOCK_M: tl.constexpr, BLOCK_N: tl.constexpr, BLOCK_DMODEL: tl.constexpr,
    IS_CAUSAL: tl.constexpr,
):
    start_m = tl.program_id(0)
    off_hz = tl.program_id(1)
    off_z = off_hz // H
    off_h = off_hz % H

    offs_m = start_m * BLOCK_M + tl.arange(0, BLOCK_M)
    offs_d = tl.arange(0, BLOCK_DMODEL)

    q_offset = off_z * stride_qz + off_h * stride_qh
    o_offset = off_z * stride_oz + off_h * stride_oh

    q_ptrs = Q + q_offset + offs_m[:, None] * stride_qm + offs_d[None, :] * stride_qd
    q_mask = offs_m[:, None] < N_CTX
    q = tl.load(q_ptrs, mask=q_mask, other=0.0)

    do_ptrs = dO + o_offset + offs_m[:, None] * stride_om + offs_d[None, :] * stride_od
    do_mask = offs_m[:, None] < N_CTX
    do = tl.load(do_ptrs, mask=do_mask, other=0.0)

    l_ptrs = L + off_hz * N_CTX + offs_m
    l_mask = offs_m < N_CTX
    L_i = tl.load(l_ptrs, mask=l_mask, other=0.0)

    d_ptrs = D + off_hz * N_CTX + offs_m
    delta_i = tl.load(d_ptrs, mask=l_mask, other=0.0)

    dq = tl.zeros([BLOCK_M, BLOCK_DMODEL], dtype=tl.float32)

    if IS_CAUSAL:
        hi = tl.cdiv((start_m + 1) * BLOCK_M, BLOCK_N)
    else:
        hi = tl.cdiv(N_CTX, BLOCK_N)

    k_offset = off_z * stride_kz + off_h * stride_kh
    v_offset = off_z * stride_vz + off_h * stride_vh

    for start_n in range(0, hi * BLOCK_N, BLOCK_N):
        start_n = tl.multiple_of(start_n, BLOCK_N)
        offs_n = start_n + tl.arange(0, BLOCK_N)

        k_ptrs = K + k_offset + offs_n[:, None] * stride_kn + offs_d[None, :] * stride_kd
        k_mask = offs_n[:, None] < N_CTX
        k_block = tl.load(k_ptrs, mask=k_mask, other=0.0)

        v_ptrs = V + v_offset + offs_n[:, None] * stride_vn + offs_d[None, :] * stride_vd
        v_mask = offs_n[:, None] < N_CTX
        v_block = tl.load(v_ptrs, mask=v_mask, other=0.0)

        s = tl.dot(q, tl.trans(k_block)) * scale

        if IS_CAUSAL:
            causal = offs_m[:, None] >= offs_n[None, :]
            s = tl.where(causal, s, float('-inf'))
        n_mask = offs_n[None, :] < N_CTX
        s = tl.where(n_mask, s, float('-inf'))
        m_mask = offs_m[:, None] < N_CTX
        s = tl.where(m_mask, s, float('-inf'))

        p = tl.exp(s - L_i[:, None])

        dp = tl.dot(do, tl.trans(v_block))
        ds = p * (dp - delta_i[:, None])

        dq += tl.dot(ds.to(q.dtype), k_block) * scale

    dq_offset = off_z * stride_dqz + off_h * stride_dqh
    dq_ptrs = dQ + dq_offset + offs_m[:, None] * stride_dqm + offs_d[None, :] * stride_dqd
    tl.store(dq_ptrs, dq.to(dQ.dtype.element_ty), mask=q_mask)


# ---------------------------------------------------------------------------
# Враппер backward
# ---------------------------------------------------------------------------
def flash_attn_backward(dout, q, k, v, out, L, causal=True):
    B, H, N, D = q.shape
    assert D in {16, 32, 64, 128, 256}

    dout = dout.contiguous()
    q = q.contiguous()
    k = k.contiguous()
    v = v.contiguous()
    out = out.contiguous()
    L = L.contiguous()

    delta = (dout.float() * out.float()).sum(dim=-1).contiguous()

    dq = torch.empty_like(q)
    dk = torch.empty_like(k)
    dv = torch.empty_like(v)

    scale = 1.0 / math.sqrt(D)

    # Подбор блоков под SMEM T4 (64 KB)
    if D <= 64:
        BLOCK_M, BLOCK_N = 64, 64
    elif D <= 128:
        BLOCK_M, BLOCK_N = 32, 32
    else:
        BLOCK_M, BLOCK_N = 16, 16

    # --- Кернел 1: dK, dV ---
    grid_dkdv = (triton.cdiv(N, BLOCK_N), B * H)
    _flash_attn_bwd_dkdv_kernel[grid_dkdv](
        q, k, v, dout, dk, dv, L, delta,
        q.stride(0), q.stride(1), q.stride(2), q.stride(3),
        k.stride(0), k.stride(1), k.stride(2), k.stride(3),
        v.stride(0), v.stride(1), v.stride(2), v.stride(3),
        dout.stride(0), dout.stride(1), dout.stride(2), dout.stride(3),
        dk.stride(0), dk.stride(1), dk.stride(2), dk.stride(3),
        dv.stride(0), dv.stride(1), dv.stride(2), dv.stride(3),
        B, H, N, scale,
        BLOCK_M=BLOCK_M, BLOCK_N=BLOCK_N, BLOCK_DMODEL=D,
        IS_CAUSAL=causal,
    )

    # --- Кернел 2: dQ ---
    grid_dq = (triton.cdiv(N, BLOCK_M), B * H)
    _flash_attn_bwd_dq_kernel[grid_dq](
        q, k, v, dout, dq, L, delta,
        q.stride(0), q.stride(1), q.stride(2), q.stride(3),
        k.stride(0), k.stride(1), k.stride(2), k.stride(3),
        v.stride(0), v.stride(1), v.stride(2), v.stride(3),
        dout.stride(0), dout.stride(1), dout.stride(2), dout.stride(3),
        dq.stride(0), dq.stride(1), dq.stride(2), dq.stride(3),
        B, H, N, scale,
        BLOCK_M=BLOCK_M, BLOCK_N=BLOCK_N, BLOCK_DMODEL=D,
        IS_CAUSAL=causal,
    )

    return dq, dk, dv

Overwriting /content/MNNA-2026/src/backend/flash_attention_backward.py


In [35]:
#Промежуточная проверка 1
import sys, importlib
sys.path.insert(0, '/content/MNNA-2026')

import src.backend.flash_attention_backward as fab
importlib.reload(fab)
print("Модуль flash_attention_backward импортируется.")
print("Функция flash_attn_backward:", fab.flash_attn_backward)

Модуль flash_attention_backward импортируется.
Функция flash_attn_backward: <function flash_attn_backward at 0x7cbc3d6a5bc0>


In [4]:
#Промежуточная проверка 2
import sys, importlib
sys.path.insert(0, '/content/MNNA-2026')
import torch, triton, math
import src.backend.flash_attention_backward as fab
importlib.reload(fab)
from src.backend.flash_attention_backward import _flash_attn_bwd_dkdv_kernel

torch.manual_seed(0)

B, H, N, D = 1, 1, 128, 64
device = 'cuda'
q = torch.randn(B, H, N, D, device=device, dtype=torch.float16)
k = torch.randn(B, H, N, D, device=device, dtype=torch.float16)
v = torch.randn(B, H, N, D, device=device, dtype=torch.float16)
dout = torch.randn(B, H, N, D, device=device, dtype=torch.float16)

# Эталон forward (fp32)
scale = 1.0 / math.sqrt(D)
s = (q.float() @ k.float().transpose(-2, -1)) * scale
mask = torch.tril(torch.ones(N, N, device=device, dtype=torch.bool))
s = s.masked_fill(~mask, float('-inf'))
p = torch.softmax(s, dim=-1)
out = (p @ v.float())
L = torch.logsumexp(s, dim=-1)
delta = (dout.float() * out).sum(dim=-1)

# Эталон backward
dv_ref = (p.transpose(-2, -1) @ dout.float())
dp = dout.float() @ v.float().transpose(-2, -1)
ds = p * (dp - delta[..., None])
dk_ref = (ds.transpose(-2, -1) @ q.float()) * scale

# Наш кернел
dk = torch.empty_like(k)
dv = torch.empty_like(v)

BLOCK_M, BLOCK_N = 64, 64          # <-- ВАЖНО: 64, 64, не 128!
grid = (triton.cdiv(N, BLOCK_N), B * H)

_flash_attn_bwd_dkdv_kernel[grid](
    q.contiguous(), k.contiguous(), v.contiguous(),
    dout.contiguous(), dk, dv,
    L.contiguous().float(), delta.contiguous().float(),
    q.stride(0), q.stride(1), q.stride(2), q.stride(3),
    k.stride(0), k.stride(1), k.stride(2), k.stride(3),
    v.stride(0), v.stride(1), v.stride(2), v.stride(3),
    dout.stride(0), dout.stride(1), dout.stride(2), dout.stride(3),
    dk.stride(0), dk.stride(1), dk.stride(2), dk.stride(3),
    dv.stride(0), dv.stride(1), dv.stride(2), dv.stride(3),
    B, H, N, scale,
    BLOCK_M=BLOCK_M, BLOCK_N=BLOCK_N, BLOCK_DMODEL=D,
    IS_CAUSAL=True,
)

print("dv_ref sample:", dv_ref.flatten()[:4].tolist())
print("dv     sample:", dv.float().flatten()[:4].tolist())
print("dk_ref sample:", dk_ref.flatten()[:4].tolist())
print("dk     sample:", dk.float().flatten()[:4].tolist())

torch.testing.assert_close(dk.float(), dk_ref, atol=1e-2, rtol=1e-2)
torch.testing.assert_close(dv.float(), dv_ref, atol=1e-2, rtol=1e-2)
print("✅ dK и dV численно совпадают с эталоном.")

dv_ref sample: [1.2533674240112305, -1.0466344356536865, -0.22195684909820557, -0.5176795125007629]
dv     sample: [1.2529296875, -1.046875, -0.22216796875, -0.517578125]
dk_ref sample: [0.1156996414065361, -0.24853146076202393, 0.5722889304161072, -1.7607970237731934]
dk     sample: [0.11566162109375, -0.24853515625, 0.572265625, -1.7607421875]
✅ dK и dV численно совпадают с эталоном.


In [6]:
#Промежуточная проверка 3
import sys, importlib
sys.path.insert(0, '/content/MNNA-2026')
import torch, math
import src.backend.flash_attention_backward as fab
importlib.reload(fab)
from src.backend.flash_attention_backward import flash_attn_backward

torch.manual_seed(0)

B, H, N, D = 1, 1, 128, 64
device = 'cuda'
q = torch.randn(B, H, N, D, device=device, dtype=torch.float16)
k = torch.randn(B, H, N, D, device=device, dtype=torch.float16)
v = torch.randn(B, H, N, D, device=device, dtype=torch.float16)
dout = torch.randn(B, H, N, D, device=device, dtype=torch.float16)

scale = 1.0 / math.sqrt(D)
s = (q.float() @ k.float().transpose(-2, -1)) * scale
mask = torch.tril(torch.ones(N, N, device=device, dtype=torch.bool))
s = s.masked_fill(~mask, float('-inf'))
p = torch.softmax(s, dim=-1)
out = (p @ v.float())
L = torch.logsumexp(s, dim=-1)
delta = (dout.float() * out).sum(dim=-1)

# Эталон
dv_ref = (p.transpose(-2, -1) @ dout.float())
dp = dout.float() @ v.float().transpose(-2, -1)
ds = p * (dp - delta[..., None])
dk_ref = (ds.transpose(-2, -1) @ q.float()) * scale
dq_ref = (ds @ k.float()) * scale

# Наш backward
dq, dk, dv = flash_attn_backward(dout, q, k, v, out.to(torch.float16), L, causal=True)

print("dq_ref sample:", dq_ref.flatten()[:4].tolist())
print("dq     sample:", dq.float().flatten()[:4].tolist())

torch.testing.assert_close(dq.float(), dq_ref, atol=1e-2, rtol=1e-2)
torch.testing.assert_close(dk.float(), dk_ref, atol=1e-2, rtol=1e-2)
torch.testing.assert_close(dv.float(), dv_ref, atol=1e-2, rtol=1e-2)
print("✅ dQ, dK, dV численно совпадают с эталоном.")

dq_ref sample: [0.0, 0.0, 0.0, 0.0]
dq     sample: [0.0, 0.0, 0.0, 0.0]
✅ dQ, dK, dV численно совпадают с эталоном.


In [8]:
print("dq_ref: max abs =", dq_ref.abs().max().item())
print("dq    : max abs =", dq.float().abs().max().item())
print("dk_ref: max abs =", dk_ref.abs().max().item())
print("dk    : max abs =", dk.float().abs().max().item())
print("dv_ref: max abs =", dv_ref.abs().max().item())
print("dv    : max abs =", dv.float().abs().max().item())

# Относительная ошибка в норме L2
for name, a, b in [("dq", dq.float(), dq_ref), ("dk", dk.float(), dk_ref), ("dv", dv.float(), dv_ref)]:
    rel = (a - b).norm() / b.norm()
    print(f"{name}: ||diff||/||ref|| = {rel.item():.4e}")

dq_ref: max abs = 2.9715654850006104
dq    : max abs = 2.97265625
dk_ref: max abs = 1.7756104469299316
dk    : max abs = 1.775390625
dv_ref: max abs = 3.5063211917877197
dv    : max abs = 3.505859375
dq: ||diff||/||ref|| = 3.0930e-04
dk: ||diff||/||ref|| = 3.2290e-04
dv: ||diff||/||ref|| = 2.7378e-04


In [7]:
def run_test(N, D, causal, B=2, H=4):
    q = torch.randn(B, H, N, D, device='cuda', dtype=torch.float16)
    k = torch.randn(B, H, N, D, device='cuda', dtype=torch.float16)
    v = torch.randn(B, H, N, D, device='cuda', dtype=torch.float16)
    dout = torch.randn(B, H, N, D, device='cuda', dtype=torch.float16)

    scale = 1.0 / math.sqrt(D)
    s = (q.float() @ k.float().transpose(-2, -1)) * scale
    if causal:
        m = torch.tril(torch.ones(N, N, device='cuda', dtype=torch.bool))
        s = s.masked_fill(~m, float('-inf'))
    p = torch.softmax(s, dim=-1)
    out = (p @ v.float())
    L = torch.logsumexp(s, dim=-1)
    delta = (dout.float() * out).sum(dim=-1)

    dv_ref = (p.transpose(-2, -1) @ dout.float())
    dp = dout.float() @ v.float().transpose(-2, -1)
    ds = p * (dp - delta[..., None])
    dk_ref = (ds.transpose(-2, -1) @ q.float()) * scale
    dq_ref = (ds @ k.float()) * scale

    dq, dk, dv = flash_attn_backward(dout, q, k, v, out.to(torch.float16), L, causal=causal)

    torch.testing.assert_close(dq.float(), dq_ref, atol=2e-2, rtol=2e-2)
    torch.testing.assert_close(dk.float(), dk_ref, atol=2e-2, rtol=2e-2)
    torch.testing.assert_close(dv.float(), dv_ref, atol=2e-2, rtol=2e-2)
    print(f"  OK: N={N}, D={D}, causal={causal}")

print("=== Backward тесты ===")
run_test(N=128, D=64, causal=True)
run_test(N=128, D=64, causal=False)
run_test(N=512, D=64, causal=True)
run_test(N=333, D=64, causal=True)   # N не кратен блоку
run_test(N=256, D=128, causal=True)
print("Все тесты backward пройдены.")

=== Backward тесты ===
  OK: N=128, D=64, causal=True
  OK: N=128, D=64, causal=False
  OK: N=512, D=64, causal=True
  OK: N=333, D=64, causal=True
  OK: N=256, D=128, causal=True
Все тесты backward пройдены.


In [9]:
%%writefile /content/MNNA-2026/src/backend/flash_attention.py
import torch
import torch.nn as nn

from src.backend.flash_attention_forward import flash_attn_forward
from src.backend.flash_attention_backward import flash_attn_backward


class FlashAttnFunc(torch.autograd.Function):
    """
    Кастомная autograd-функция для Flash Attention.

    forward:  вызывает наш forward-кернел, сохраняет q, k, v, out, L
    backward: вызывает наш backward-кернел (два кернела внутри)
    """
    @staticmethod
    def forward(ctx, q, k, v, causal):
        out, L = flash_attn_forward(q, k, v, causal=causal)
        ctx.save_for_backward(q, k, v, out, L)
        ctx.causal = causal
        return out

    @staticmethod
    def backward(ctx, dout):
        q, k, v, out, L = ctx.saved_tensors
        dq, dk, dv = flash_attn_backward(dout, q, k, v, out, L, causal=ctx.causal)
        return dq, dk, dv, None   # None для causal (это bool, не тензор)


class FlashAttention(nn.Module):
    """
    nn.Module-обёртка над FlashAttnFunc.

    Args:
        causal: bool — применять ли causal-маску
    """
    def __init__(self, causal: bool = True):
        super().__init__()
        self.causal = causal

    def forward(self, q, k, v):
        return FlashAttnFunc.apply(q, k, v, self.causal)

Overwriting /content/MNNA-2026/src/backend/flash_attention.py


In [10]:
import sys, importlib
sys.path.insert(0, '/content/MNNA-2026')
import torch, math

import src.backend.flash_attention as fa
importlib.reload(fa)
from src.backend.flash_attention import FlashAttnFunc, FlashAttention


def test_autograd(N, D, causal, B=2, H=4, atol=2e-2, rtol=2e-2):
    torch.manual_seed(0)
    device = 'cuda'
    q = torch.randn(B, H, N, D, device=device, dtype=torch.float16, requires_grad=True)
    k = torch.randn(B, H, N, D, device=device, dtype=torch.float16, requires_grad=True)
    v = torch.randn(B, H, N, D, device=device, dtype=torch.float16, requires_grad=True)
    dout = torch.randn(B, H, N, D, device=device, dtype=torch.float16)

    # --- Наш Flash Attention через autograd ---
    out_fa = FlashAttnFunc.apply(q, k, v, causal)
    out_fa.backward(dout)
    dq_fa, dk_fa, dv_fa = q.grad.clone(), k.grad.clone(), v.grad.clone()

    # --- Эталон через наивную torch-реализацию ---
    q2 = q.detach().clone().requires_grad_(True)
    k2 = k.detach().clone().requires_grad_(True)
    v2 = v.detach().clone().requires_grad_(True)

    scale = 1.0 / math.sqrt(D)
    s = (q2.float() @ k2.float().transpose(-2, -1)) * scale
    if causal:
        m = torch.tril(torch.ones(N, N, device=device, dtype=torch.bool))
        s = s.masked_fill(~m, float('-inf'))
    p = torch.softmax(s, dim=-1)
    out_ref = p @ v2.float()
    out_ref.to(torch.float16).backward(dout)
    dq_ref, dk_ref, dv_ref = q2.grad.clone(), k2.grad.clone(), v2.grad.clone()

    # Сравнение
    torch.testing.assert_close(dq_fa.float(), dq_ref.float(), atol=atol, rtol=rtol)
    torch.testing.assert_close(dk_fa.float(), dk_ref.float(), atol=atol, rtol=rtol)
    torch.testing.assert_close(dv_fa.float(), dv_ref.float(), atol=atol, rtol=rtol)
    # И сам выход
    torch.testing.assert_close(out_fa.float(), out_ref.to(torch.float16).float(), atol=atol, rtol=rtol)
    print(f"  OK: N={N}, D={D}, causal={causal}")


print("=== Autograd тесты (через .backward()) ===")
test_autograd(N=128, D=64, causal=True)
test_autograd(N=128, D=64, causal=False)
test_autograd(N=512, D=64, causal=True)
test_autograd(N=333, D=64, causal=True)
test_autograd(N=256, D=128, causal=True)

print()
print("=== Тест nn.Module ===")
module = FlashAttention(causal=True).cuda()
q = torch.randn(2, 4, 256, 64, device='cuda', dtype=torch.float16, requires_grad=True)
k = torch.randn(2, 4, 256, 64, device='cuda', dtype=torch.float16, requires_grad=True)
v = torch.randn(2, 4, 256, 64, device='cuda', dtype=torch.float16, requires_grad=True)
out = module(q, k, v)
loss = out.float().sum()
loss.backward()
print(f"  nn.Module работает, out.shape = {out.shape}")
print(f"  q.grad.max() = {q.grad.abs().max().item():.4f}, "
      f"k.grad.max() = {k.grad.abs().max().item():.4f}, "
      f"v.grad.max() = {v.grad.abs().max().item():.4f}")
print()
print("✅ Все тесты autograd и nn.Module пройдены.")

=== Autograd тесты (через .backward()) ===
  OK: N=128, D=64, causal=True
  OK: N=128, D=64, causal=False
  OK: N=512, D=64, causal=True
  OK: N=333, D=64, causal=True
  OK: N=256, D=128, causal=True

=== Тест nn.Module ===
  nn.Module работает, out.shape = torch.Size([2, 4, 256, 64])
  q.grad.max() = 2.2305, k.grad.max() = 4.3555, v.grad.max() = 6.4453

✅ Все тесты autograd и nn.Module пройдены.


In [12]:
import sys, importlib
sys.path.insert(0, '/content/MNNA-2026')
import torch, time, math
import torch.nn.functional as F

import src.backend.flash_attention as fa
importlib.reload(fa)
from src.backend.flash_attention import FlashAttnFunc


def bench(fn, warmup=5, iters=20):
    """Возвращает среднее время в мс."""
    for _ in range(warmup):
        fn()
    torch.cuda.synchronize()
    t0 = time.perf_counter()
    for _ in range(iters):
        fn()
    torch.cuda.synchronize()
    return (time.perf_counter() - t0) / iters * 1000  # ms


def make_inputs(B, H, N, D, device='cuda'):
    q = torch.randn(B, H, N, D, device=device, dtype=torch.float16, requires_grad=True)
    k = torch.randn(B, H, N, D, device=device, dtype=torch.float16, requires_grad=True)
    v = torch.randn(B, H, N, D, device=device, dtype=torch.float16, requires_grad=True)
    return q, k, v


def run_ours(B, H, N, D, causal):
    q, k, v = make_inputs(B, H, N, D)
    def fn():
        out = FlashAttnFunc.apply(q, k, v, causal)
        out.sum().backward()
        q.grad = k.grad = v.grad = None
    return bench(fn)


def run_sdpa(B, H, N, D, causal):
    q, k, v = make_inputs(B, H, N, D)
    def fn():
        out = F.scaled_dot_product_attention(q, k, v, is_causal=causal)
        out.sum().backward()
        q.grad = k.grad = v.grad = None
    return bench(fn)


def peak_memory_ours(B, H, N, D, causal):
    q, k, v = make_inputs(B, H, N, D)
    torch.cuda.reset_peak_memory_stats()
    out = FlashAttnFunc.apply(q, k, v, causal)
    out.sum().backward()
    return torch.cuda.max_memory_allocated() / 1024**2  # MB


def peak_memory_sdpa(B, H, N, D, causal):
    q, k, v = make_inputs(B, H, N, D)
    torch.cuda.reset_peak_memory_stats()
    out = F.scaled_dot_product_attention(q, k, v, is_causal=causal)
    out.sum().backward()
    return torch.cuda.max_memory_allocated() / 1024**2

In [13]:
print("=== Скорость forward+backward (мс) ===")
print(f"{'N':>6} | {'Ours':>10} | {'SDPA':>10} | {'Speedup':>10}")
print("-" * 44)

B, H, D = 2, 4, 64
for N in [256, 512, 1024, 2048]:
    t_ours = run_ours(B, H, N, D, causal=True)
    t_sdpa = run_sdpa(B, H, N, D, causal=True)
    speedup = t_sdpa / t_ours
    print(f"{N:>6} | {t_ours:>8.2f}ms | {t_sdpa:>8.2f}ms | {speedup:>8.2f}x")

=== Скорость forward+backward (мс) ===
     N |       Ours |       SDPA |    Speedup
--------------------------------------------
   256 |     4.42ms |     0.48ms |     0.11x
   512 |    10.19ms |     0.48ms |     0.05x
  1024 |    34.65ms |     0.58ms |     0.02x
  2048 |   130.50ms |     1.87ms |     0.01x


In [14]:
print("\n=== Пиковая память forward+backward (MB) ===")
print(f"{'N':>6} | {'Ours':>10} | {'SDPA':>10} | {'Ratio':>10}")
print("-" * 44)

for N in [256, 512, 1024, 2048]:
    m_ours = peak_memory_ours(B, H, N, D, causal=True)
    m_sdpa = peak_memory_sdpa(B, H, N, D, causal=True)
    ratio = m_ours / m_sdpa
    print(f"{N:>6} | {m_ours:>8.2f}MB | {m_sdpa:>8.2f}MB | {ratio:>8.2f}x")


=== Пиковая память forward+backward (MB) ===
     N |       Ours |       SDPA |      Ratio
--------------------------------------------
   256 |    21.31MB |    21.07MB |     1.01x
   512 |    24.07MB |    23.58MB |     1.02x
  1024 |    29.58MB |    28.62MB |     1.03x
  2048 |    40.61MB |    38.68MB |     1.05x


In [11]:
import sys
sys.path.insert(0, '/content/MNNA-2026')
import torch, triton
from src.backend.flash_attention_backward import _flash_attn_bwd_dkdv_kernel

def try_block(BM, BN):
    B, H, N, D = 1, 1, 256, 64
    q = torch.randn(B, H, N, D, device='cuda', dtype=torch.float16).contiguous()
    k, v = q.clone(), q.clone()
    do = q.clone()
    L = torch.zeros(B, H, N, device='cuda', dtype=torch.float32)
    delta = torch.zeros(B, H, N, device='cuda', dtype=torch.float32)
    dk, dv = torch.empty_like(k), torch.empty_like(v)
    scale = 1.0 / (D ** 0.5)

    grid = (triton.cdiv(N, BN), B * H)
    try:
        _flash_attn_bwd_dkdv_kernel[grid](
            q, k, v, do, dk, dv, L, delta,
            q.stride(0), q.stride(1), q.stride(2), q.stride(3),
            k.stride(0), k.stride(1), k.stride(2), k.stride(3),
            v.stride(0), v.stride(1), v.stride(2), v.stride(3),
            do.stride(0), do.stride(1), do.stride(2), do.stride(3),
            dk.stride(0), dk.stride(1), dk.stride(2), dk.stride(3),
            dv.stride(0), dv.stride(1), dv.stride(2), dv.stride(3),
            B, H, N, scale,
            BLOCK_M=BM, BLOCK_N=BN, BLOCK_DMODEL=D,
            IS_CAUSAL=True,
        )
        print(f"BM={BM:3d}, BN={BN:3d} -> OK")
    except Exception as e:
        print(f"BM={BM:3d}, BN={BN:3d} -> FAIL ({type(e).__name__})")

for BM, BN in [(16,16), (32,16), (32,32), (64,16), (64,32), (64,64), (128,16), (128,32)]:
    try_block(BM, BN)

BM= 16, BN= 16 -> OK
BM= 32, BN= 16 -> OK
BM= 32, BN= 32 -> OK
BM= 64, BN= 16 -> OK
BM= 64, BN= 32 -> OK
BM= 64, BN= 64 -> OK
BM=128, BN= 16 -> FAIL (OutOfResources)
BM=128, BN= 32 -> FAIL (OutOfResources)
